In [ ]:
%run "../scripts/load_daily_variables.py"

In [ ]:
import warnings

import numpy as np
import pandas as pd

# TC2000 definitions: https://help.tc2000.com/m/69404/c/213566
# Prices and volume always come from the DataFrames in `symbols`. The Finviz
# snapshot is used only because SymbolData has no exchange/member attribute.


def _find_column(frame, candidates):
    """Return the first case-insensitive matching column name."""
    lookup = {str(column).casefold(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    return None


def _numeric_market_cap(value):
    """Convert numeric or K/M/B/T-suffixed market-cap values to dollars."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).strip().upper().replace(",", "").replace("$", "")
    multiplier = {"K": 1e3, "M": 1e6, "B": 1e9, "T": 1e12}.get(text[-1:], 1.0)
    if multiplier != 1.0:
        text = text[:-1]
    try:
        return float(text) * multiplier
    except ValueError:
        return np.nan


def _field_panel(field):
    """Align one OHLCV field from every SymbolData DataFrame by session."""
    series = {}
    for ticker, stock in symbols.items():
        frame = stock.df
        column = _find_column(frame, (field, field.capitalize(), field.upper()))
        if column is None or frame.empty:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        dates = pd.DatetimeIndex(pd.to_datetime(values.index, utc=True)).tz_localize(None).normalize()
        values = pd.Series(values.to_numpy(), index=dates, name=ticker)
        series[ticker] = values.groupby(level=0).last()
    if not series:
        raise ValueError(f"No {field!r} data was found in symbols[*].df")
    return pd.concat(series, axis=1).sort_index()


close = _field_panel("close")
high = _field_panel("high").reindex(index=close.index, columns=close.columns)
low = _field_panel("low").reindex(index=close.index, columns=close.columns)
volume = _field_panel("volume").reindex(index=close.index, columns=close.columns)

# Build exchange and market-cap universes. TC2000's High Cap lists rank US
# common stocks by market cap. Nasdaq 100 constituents are not saved by the
# producer, so its top-100 Nasdaq market-cap proxy is called out explicitly.
metadata = results_finvizsearch.copy()
ticker_column = _find_column(metadata, ("Ticker", "Symbol"))
exchange_column = _find_column(metadata, ("Exchange",))
market_cap_column = _find_column(metadata, ("Market Cap.", "Market Cap", "Market Cap ($)"))
if ticker_column is None or exchange_column is None:
    raise ValueError("results_finvizsearch must contain Ticker/Symbol and Exchange columns")

metadata[ticker_column] = metadata[ticker_column].astype(str).str.upper()
metadata = metadata.drop_duplicates(ticker_column, keep="last").set_index(ticker_column)
exchange = metadata[exchange_column].fillna("").astype(str).str.upper()
market_cap = (
    metadata[market_cap_column].map(_numeric_market_cap)
    if market_cap_column is not None
    else pd.Series(dtype=float)
)
# Prefer a populated SymbolData.market_cap if a later producer adds it.
for ticker, stock in symbols.items():
    value = getattr(stock, "market_cap", None)
    if value is not None and pd.notna(value):
        market_cap.loc[ticker] = float(value)

available = set(close.columns)
nyse_symbols = sorted(available & set(exchange[exchange.str.contains("NYSE", regex=False)].index))
nasdaq_symbols = sorted(available & set(exchange[exchange.str.contains("NASDAQ", regex=False)].index))
us_common_symbols = sorted(
    available
    & set(exchange[exchange.str.contains("NYSE|NASDAQ|AMEX", regex=True)].index)
)
if not nyse_symbols or not nasdaq_symbols:
    raise ValueError("The exchange metadata did not produce both NYSE and Nasdaq universes")

ranked_us_symbols = [
    ticker for ticker in market_cap.reindex(us_common_symbols).sort_values(ascending=False).dropna().index
]
high_cap_1000_symbols = ranked_us_symbols[:1000]
mid_cap_2000_symbols = ranked_us_symbols[1000:3000]
high_cap_3000_symbols = ranked_us_symbols[:3000]
nasdaq_100_symbols = [
    ticker for ticker in market_cap.reindex(nasdaq_symbols).sort_values(ascending=False).dropna().index[:100]
]
if len(ranked_us_symbols) < 3000:
    warnings.warn(
        f"Only {len(ranked_us_symbols)} US symbols have market-cap metadata; "
        "High/Mid Cap universes are truncated.",
        stacklevel=2,
    )


def _breadth(universe):
    prices = close.reindex(columns=universe)
    previous = prices.shift(1)
    valid = prices.notna() & previous.notna()
    advancing = ((prices > previous) & valid).sum(axis=1)
    declining = ((prices < previous) & valid).sum(axis=1)
    unchanged = ((prices == previous) & valid).sum(axis=1)
    total = valid.sum(axis=1).replace(0, np.nan)
    advancing_volume = volume.reindex(columns=universe).where((prices > previous) & valid).sum(axis=1)
    declining_volume = volume.reindex(columns=universe).where((prices < previous) & valid).sum(axis=1)
    return pd.DataFrame(
        {
            "advancing": advancing,
            "declining": declining,
            "unchanged": unchanged,
            "total": total,
            "net_advances": advancing - declining,
            "advancing_volume": advancing_volume,
            "declining_volume": declining_volume,
        }
    )


def _advance_decline_line(universe):
    breadth = _breadth(universe)
    daily_value = 100 * breadth["net_advances"].div(breadth["total"])
    return daily_value.fillna(0).cumsum().rename("advance_decline_line")


def _new_high_low_counts(universe, sessions):
    universe_high = high.reindex(columns=universe)
    universe_low = low.reindex(columns=universe)
    rolling_high = universe_high.rolling(sessions, min_periods=sessions).max()
    rolling_low = universe_low.rolling(sessions, min_periods=sessions).min()
    new_highs = (universe_high >= rolling_high).sum(axis=1)
    new_lows = (universe_low <= rolling_low).sum(axis=1)
    return new_highs, new_lows


def _new_high_low_ratio(sessions):
    new_highs, new_lows = _new_high_low_counts(nyse_symbols, sessions)
    denominator = (new_highs + new_lows).replace(0, np.nan)
    return (100 * new_highs.div(denominator)).rename("new_high_new_low_ratio")


def _percentage_relative_to_pma(period, channels=0, direction="above"):
    prices = close.reindex(columns=nyse_symbols)
    pma = prices.rolling(period, min_periods=period).mean()
    deviation = prices.rolling(period, min_periods=period).std(ddof=0)
    boundary = pma + channels * deviation if direction == "above" else pma - channels * deviation
    eligible = prices.notna() & boundary.notna()
    matches = (prices > boundary) if direction == "above" else (prices < boundary)
    return (100 * (matches & eligible).sum(axis=1).div(eligible.sum(axis=1).replace(0, np.nan)))


nyse_breadth = _breadth(nyse_symbols)

# T2100, T2125-T2129: cumulative advancing percentage minus declining percentage.
advance_decline_line_nyse = _advance_decline_line(nyse_symbols)
advance_decline_line_nasdaq = _advance_decline_line(nasdaq_symbols)
advance_decline_line_nasdaq_100 = _advance_decline_line(nasdaq_100_symbols)
advance_decline_line_high_cap_1000 = _advance_decline_line(high_cap_1000_symbols)
advance_decline_line_mid_cap_2000 = _advance_decline_line(mid_cap_2000_symbols)
advance_decline_line_high_cap_3000 = _advance_decline_line(high_cap_3000_symbols)

# T2101: absolute five-session net advances as a percentage of NYSE issues.
absolute_breadth_index = (
    100
    * nyse_breadth["net_advances"].rolling(5, min_periods=5).sum().abs()
    .div(nyse_breadth["total"])
).rename("absolute_breadth_index")

# T2102: cumulative signed square root of (advances/unchanged - declines/unchanged).
bolton_tremblay_increment = np.sign(nyse_breadth["net_advances"]) * np.sqrt(
    nyse_breadth["net_advances"].abs().div(nyse_breadth["unchanged"].replace(0, np.nan))
)
bolton_tremblay_indicator = bolton_tremblay_increment.fillna(0).cumsum().rename(
    "bolton_tremblay_indicator"
)

# T2103: ten-session SMA of advances/(advances + declines), on a 0-100 scale.
zweig_breadth_thrust = (
    100
    * nyse_breadth["advancing"]
    .div((nyse_breadth["advancing"] + nyse_breadth["declining"]).replace(0, np.nan))
    .rolling(10, min_periods=10)
    .mean()
).rename("zweig_breadth_thrust")

# T2104: cumulative advancing-issue volume minus declining-issue volume.
cumulative_volume_index = (
    nyse_breadth["advancing_volume"] - nyse_breadth["declining_volume"]
).fillna(0).cumsum().rename("cumulative_volume_index")

# T2105 uses 52-week (260-session) new highs/lows and the smaller percentage.
new_highs_52_week, new_lows_52_week = _new_high_low_counts(nyse_symbols, 260)
high_low_logic_index = (
    100
    * pd.concat([new_highs_52_week, new_lows_52_week], axis=1).min(axis=1)
    .div(nyse_breadth["total"])
).rename("high_low_logic_index")

# T2106 and T2118: 19-day EMA - 39-day EMA of net advances, then cumulative sum.
mcclellan_oscillator = (
    nyse_breadth["net_advances"].ewm(span=19, adjust=False).mean()
    - nyse_breadth["net_advances"].ewm(span=39, adjust=False).mean()
).rename("mcclellan_oscillator")
mcclellan_summation_index = mcclellan_oscillator.fillna(0).cumsum().rename(
    "mcclellan_summation_index"
)

# T2117, T2120-T2122: new highs / (new highs + new lows), on a 0-100 scale.
new_high_new_low_ratio_52_week = _new_high_low_ratio(260)
new_high_new_low_ratio_26_week = _new_high_low_ratio(130)
new_high_new_low_ratio_13_week = _new_high_low_ratio(65)
new_high_new_low_ratio_4_week = _new_high_low_ratio(20)

# T2123's historical name says cumulative, but TC2000 defines it as today's
# count of four-week new highs minus today's count of four-week new lows.
new_highs_4_week, new_lows_4_week = _new_high_low_counts(nyse_symbols, 20)
cumulative_4_week_new_high_low = (new_highs_4_week - new_lows_4_week).rename(
    "cumulative_4_week_new_high_low"
)

# T2107-T2116: NYSE percentages relative to simple PMAs and population std-dev channels.
percentage_of_stocks_above_200_day_pma = _percentage_relative_to_pma(200)
percentage_of_stocks_above_40_day_pma = _percentage_relative_to_pma(40)
percentage_of_stocks_1_channel_above_200_day_pma = _percentage_relative_to_pma(200, 1, "above")
percentage_of_stocks_1_channel_above_40_day_pma = _percentage_relative_to_pma(40, 1, "above")
percentage_of_stocks_1_channel_below_200_day_pma = _percentage_relative_to_pma(200, 1, "below")
percentage_of_stocks_1_channel_below_40_day_pma = _percentage_relative_to_pma(40, 1, "below")
percentage_of_stocks_2_channels_above_200_day_pma = _percentage_relative_to_pma(200, 2, "above")
percentage_of_stocks_2_channels_above_40_day_pma = _percentage_relative_to_pma(40, 2, "above")
percentage_of_stocks_2_channels_below_200_day_pma = _percentage_relative_to_pma(200, 2, "below")
percentage_of_stocks_2_channels_below_40_day_pma = _percentage_relative_to_pma(40, 2, "below")

# One aligned table is convenient for comparison while the requested text-name
# variables above remain directly available as Series.
tc2000_t2_indicators = pd.DataFrame(
    {
        "absolute_breadth_index": absolute_breadth_index,
        "advance_decline_line_high_cap_1000": advance_decline_line_high_cap_1000,
        "advance_decline_line_high_cap_3000": advance_decline_line_high_cap_3000,
        "advance_decline_line_mid_cap_2000": advance_decline_line_mid_cap_2000,
        "advance_decline_line_nasdaq": advance_decline_line_nasdaq,
        "advance_decline_line_nasdaq_100": advance_decline_line_nasdaq_100,
        "advance_decline_line_nyse": advance_decline_line_nyse,
        "bolton_tremblay_indicator": bolton_tremblay_indicator,
        "cumulative_4_week_new_high_low": cumulative_4_week_new_high_low,
        "cumulative_volume_index": cumulative_volume_index,
        "high_low_logic_index": high_low_logic_index,
        "mcclellan_oscillator": mcclellan_oscillator,
        "mcclellan_summation_index": mcclellan_summation_index,
        "new_high_new_low_ratio_4_week": new_high_new_low_ratio_4_week,
        "new_high_new_low_ratio_13_week": new_high_new_low_ratio_13_week,
        "new_high_new_low_ratio_26_week": new_high_new_low_ratio_26_week,
        "new_high_new_low_ratio_52_week": new_high_new_low_ratio_52_week,
        "percentage_of_stocks_1_channel_above_200_day_pma": percentage_of_stocks_1_channel_above_200_day_pma,
        "percentage_of_stocks_1_channel_above_40_day_pma": percentage_of_stocks_1_channel_above_40_day_pma,
        "percentage_of_stocks_1_channel_below_200_day_pma": percentage_of_stocks_1_channel_below_200_day_pma,
        "percentage_of_stocks_1_channel_below_40_day_pma": percentage_of_stocks_1_channel_below_40_day_pma,
        "percentage_of_stocks_2_channels_above_200_day_pma": percentage_of_stocks_2_channels_above_200_day_pma,
        "percentage_of_stocks_2_channels_above_40_day_pma": percentage_of_stocks_2_channels_above_40_day_pma,
        "percentage_of_stocks_2_channels_below_200_day_pma": percentage_of_stocks_2_channels_below_200_day_pma,
        "percentage_of_stocks_2_channels_below_40_day_pma": percentage_of_stocks_2_channels_below_40_day_pma,
        "percentage_of_stocks_above_200_day_pma": percentage_of_stocks_above_200_day_pma,
        "percentage_of_stocks_above_40_day_pma": percentage_of_stocks_above_40_day_pma,
        "zweig_breadth_thrust": zweig_breadth_thrust,
    }
)

tc2000_t2_indicators.tail()